# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook guides you through loading, exploring, and processing the [FAIR^2 Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://sen.science/doi/10.71728/senscience.vq4a-28xa/) dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is described in Croissant schema format and available at:
`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Install mlcroissant if not already installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}\n\nIdentifier: {metadata.identifier}\nVersion: {metadata.version}")

## 2. Data Overview
Review available record sets, fields, and their IDs. All Croissant entities, including record sets and fields, are referenced by their `@id`.

Let's list all available record sets in this dataset.

In [ ]:
# List all record sets and display their @id and names
record_sets = list(metadata.record_sets)

if not record_sets:
    print("No record sets found in the Croissant schema metadata.")
else:
    print("Available Record Sets:")
    for rs in record_sets:
        print(f"  - @id: {rs.id} | name: {rs.name}")

Let's enumerate the fields within each available record set by their `@id`. This is helpful to understand what data is available and how to reference it.

In [ ]:
# For each record set, list the fields with their @id, name, and type
if not record_sets:
    print("No record sets present to enumerate fields.")
else:
    for rs in record_sets:
        print(f"\nFields for record set '{rs.name}' (@id: {rs.id}):")
        for field in rs.fields:
            col_ids = [col.id for col in getattr(field, 'columns', [])]
            print(f"  - Field @id: {field.id}, name: '{field.name}', dataType: {getattr(field, 'data_type', None)}, source column(s): {col_ids}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

👉 For this notebook, we will use the first available record set (if present) for demonstration.

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

# Extract data for all record sets
dataframes = {}
record_set_ids = [rs.id for rs in record_sets]

for rs in record_sets:
    print(f"Loading records from record set: {rs.name} (@id: {rs.id})")
    try:
        records = list(dataset.records(record_set=rs.id))
        df = pd.DataFrame(records)
        dataframes[rs.id] = df
        print(f"Loaded {len(df)} records. Columns: {df.columns.tolist()}")
    except Exception as e:
        print(f"Failed to load '{rs.id}': {e}")

# Preview the first record set's DataFrame
if record_set_ids:
    first_rs_id = record_set_ids[0]
    print(f"\nFirst few records from record set '{first_rs_id}':")
    display(dataframes[first_rs_id].head())
else:
    print("No record sets present for data extraction.")

## 4. Exploratory Data Analysis (EDA)
Apply standard data processing steps such as filtering records based on specific numeric field values, normalizing data, and grouping. All fields and columns are referenced via their `@id`.

Below, we select a numeric field (e.g., normalized protein abundance) for demonstration. Update the field IDs as needed based on dataset overview.

In [ ]:
# You may need to update these IDs based on your dataset fields!
import numpy as np

# Example: Choose the first numeric/integer/float field available
selected_record_set = None
numeric_field_id = None
group_field_id = None

for rs in record_sets:
    numeric_fields = [f for f in rs.fields if getattr(f, 'data_type', '').lower() in ('number', 'float', 'integer')]
    if numeric_fields:
        selected_record_set = rs.id
        numeric_field_id = numeric_fields[0].id
        # Find a different field for grouping (preferably categorical/text)
        text_fields = [f for f in rs.fields if f.id != numeric_field_id and getattr(f, 'data_type', '').lower() in ('text', str(None))]
        if text_fields:
            group_field_id = text_fields[0].id
        break

if not selected_record_set or not numeric_field_id:
    print("No numeric field found for demonstration. Please pick appropriate field '@id' manually.")
else:
    df = dataframes[selected_record_set]

    if numeric_field_id not in df.columns:
        print(f"The numeric field '{numeric_field_id}' is not in DataFrame columns: {df.columns.tolist()}.")
    else:
        # Filtering
        threshold = np.nanmean(df[numeric_field_id]) if np.issubdtype(df[numeric_field_id].dtype, np.number) else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records in '{selected_record_set}' with '{numeric_field_id}' > {threshold:.4f} (mean): {len(filtered_df)} records.")

        # Normalizing
        mean = filtered_df[numeric_field_id].mean()
        std = filtered_df[numeric_field_id].std()
        norm_colname = f"{numeric_field_id}_normalized"
        filtered_df[norm_colname] = (filtered_df[numeric_field_id] - mean) / std
        print(f"Top normalized rows:")
        display(filtered_df[[numeric_field_id, norm_colname]].head())

        # Grouping (if possible)
        if group_field_id and group_field_id in df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Mean of '{numeric_field_id}' grouped by '{group_field_id}' (top 5):")
            display(grouped_df.head())
        else:
            print("No suitable text/categorical field found for grouping.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Here, we plot a histogram of the selected numeric field and (if available) a bar plot by a grouping field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if selected_record_set and numeric_field_id and numeric_field_id in df.columns:
    fig, ax = plt.subplots(1, 2, figsize=(12, 5))

    # Histogram
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True, ax=ax[0])
    ax[0].set_title(f"Distribution of '{numeric_field_id}'")

    # If group_field_id is defined, show mean by group
    if group_field_id and group_field_id in df.columns:
        group_means = df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        sns.barplot(x=group_field_id, y=numeric_field_id, data=group_means, ax=ax[1])
        ax[1].set_title(f"Mean '{numeric_field_id}' by '{group_field_id}'")
        ax[1].tick_params(axis='x', rotation=30)
    else:
        ax[1].axis('off')
        ax[1].set_title('No suitable group field found')

    plt.tight_layout()
    plt.show()
else:
    print("No suitable data available for visualization.")

## 6. Conclusion
In this notebook, we demonstrated how to:
- Load and inspect the [FAIR^2 Mass Spectrometry dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa/) with `mlcroissant`.
- List available record sets and fields using their Croissant `@id`.
- Extract data and load it into Pandas DataFrames.
- Process, filter, normalize, and group data by referencing fields via their IDs.
- Visualize numeric data distributions and differences by groups.

**Next steps:** Explore specific protein features, link abundance to metadata, or integrate with external databases for advanced analysis.